# Phase 5 · C · FINAL ensemble: all runs incl. interim epochs, bagged thresholds (no training, ~20 min)
**No training, GPU not needed.** Attach `phase2-backbones` + one dataset with every `p5_probs_*.pt` you have:
N1, N1L, N2 and the new G1 / G2 files - **including `p5_probs_..._interim.pt` files** if G1/G2 have not finished
(their newest epoch is used). Every combination is ranked; for the best 6, three threshold strategies (global,
per-statute, **bagged** per-statute = averaged over 15 random dev halves) are compared by 2-fold cross-fitting, and the
2 best ensembles are written as zips. Submit `final1_*` first.

**Model (all Phase-5 notebooks):** binary relevance over the statutes seen in training. The fact description is tokenised
with the Phase-1 tokenizer and covered by up to K backbone windows; each window is pooled by a **learned attention over its
tokens** (+ mean); a window-position embedding + BiGRU over the windows + attention and **max** pooling give the document
vector -> one logit per statute. BCE with per-statute positive weights; decision thresholds chosen on dev (global vs
per-statute, compared by cross-fitting), every document gets at least one statute; output = unique, valid statute IDs.
Training uses only `train.json`; dev only for selection/thresholds.

**Run:** GPU **T4 x2** (cell 1 must print `GPUs: 2`), Internet **On**, Secret `capstone`, Add Input -> **`phase2-backbones`**.
**Save & Run All.** Never submit a `smoke_*` zip.

In [ ]:
# 1 · Settings
import os, sys, json, glob, time, random, math, gc
import numpy as np, torch
BASE = dict(
    mode="spread", K=4, size=1024,            # the whole fact description (up to 4 windows of 1,024 tokens, evenly spread if longer)
    hidden=256, dropout=0.2,                  # window vectors -> BiGRU + attention & max pooling -> one logit per statute
    batch_docs=4, accum=2, epochs=3, lr_head=1e-3, weight_decay=0.01, warmup_frac=0.05, clip=1.0,
    eval_frac=1.0, eval_docs=8, snapshot_k=2, pos_weight_cap=30.0, seed=1337, budget_hours=7.0,
)
PLAN = []
OUT = "/kaggle/working"
SMOKE_CHECK = True
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
found = glob.glob("/kaggle/input/**/phase2-backbones/manifest.json", recursive=True) or \
        glob.glob("/kaggle/input/**/manifest.json", recursive=True)
assert found, "attach the dataset phase2-backbones (Add Input -> Your Datasets)"
ROOT = os.path.dirname(found[0])
print("backbones:", ROOT, "| GPUs:", torch.cuda.device_count(), "| CPUs:", os.cpu_count(), "| runs:", [c["run"] for c in PLAN])
if PLAN and torch.cuda.device_count() == 0:
    raise SystemExit("NO GPU in this session: set Accelerator = GPU T4 x2 (or this account's GPU quota is used up)")

In [ ]:
# 2 · Hugging Face token (Kaggle secret "capstone" or "HF_TOKEN")
from huggingface_hub import login, whoami
HF_TOKEN = None
try:
    from kaggle_secrets import UserSecretsClient
    for _n in ("capstone", "HF_TOKEN"):
        try:
            HF_TOKEN = UserSecretsClient().get_secret(_n)
            if HF_TOKEN: break
        except Exception:
            pass
except ImportError:
    pass
HF_TOKEN = HF_TOKEN or os.environ.get("HF_TOKEN")
assert HF_TOKEN, "add the Hugging Face token as a Kaggle secret and attach it"
login(token=HF_TOKEN); print("Hugging Face user:", whoami(token=HF_TOKEN)["name"])

## 3 · Code
`ner_model.py` (Phase-1 backbones + full FT / LoRMA / LoRA / adapters / BitFit / partial), `ner_data.py` (its helper), `ls.py` (Phase 5: tokenisation, windows, hierarchical multi-label model, training, thresholds, submission, analysis).

In [ ]:
os.makedirs('p5src', exist_ok=True)

In [ ]:
%%writefile p5src/ner_data.py
# Phase 2 (L-NER) data utilities (pure Python, no torch):
#   gold parsing, words with exact character offsets, word-level BIO tags and span targets,
#   span decoding strategies, document-level label propagation, strict scoring, error analysis, submission.
import bisect
import re
from collections import Counter, defaultdict

# ======================================================================================================
# 1. Gold annotations
# ======================================================================================================
_START = ("start", "start_char", "start_offset", "begin", "s")
_END = ("end", "end_char", "end_offset", "stop", "e")
_LABEL = ("label", "type", "entity_type", "tag", "category")
_TEXT = ("text", "entity_text", "span", "mention", "value")
_IDS = ("id", "doc_id", "document_id", "docid")
_ENTS = ("entities", "annotations", "spans", "labels", "gold")


def _pick(d, keys):
    for k in keys:
        if k in d:
            return d[k]
    return None


def _entity(item):
    """one annotation in any common shape -> (start, end, label, text or None)"""
    if isinstance(item, (list, tuple)):
        s, e, lab = item[0], item[1], item[2]
        txt = item[3] if len(item) > 3 else None
    elif isinstance(item, dict):
        s, e, lab, txt = _pick(item, _START), _pick(item, _END), _pick(item, _LABEL), _pick(item, _TEXT)
    else:
        raise ValueError(f"unknown annotation format: {item!r}")
    return int(s), int(e), str(lab), (txt if isinstance(txt, str) else None)


def parse_gold(raw):
    """gold json (dict doc -> list, dict doc -> {entities: [...]}, or list of records) -> {doc: [(s,e,lab,txt)]}"""
    out = {}
    if isinstance(raw, list):
        for rec in raw:
            out[str(_pick(rec, _IDS))] = [_entity(x) for x in (_pick(rec, _ENTS) or [])]
        return out
    for doc, v in raw.items():
        if isinstance(v, dict):
            v = _pick(v, _ENTS) or []
        out[str(doc)] = [_entity(x) for x in v]
    return out


def parse_texts(raw):
    """text json -> {doc: text}"""
    if isinstance(raw, list):
        return {str(_pick(r, _IDS)): _pick(r, ("text", "content", "document")) for r in raw}
    return {str(k): (v if isinstance(v, str) else _pick(v, ("text", "content", "document"))) for k, v in raw.items()}


def end_convention(texts, gold):
    """Is `end` exclusive (text[s:e] == entity text) or inclusive (text[s:e+1])? Decided by the data."""
    exc = inc = total = 0
    bad = []
    for doc, ents in gold.items():
        t = texts.get(doc)
        if t is None:
            continue
        for s, e, lab, txt in ents:
            if txt is None:
                continue
            total += 1
            if t[s:e] == txt:
                exc += 1
            elif t[s:e + 1] == txt:
                inc += 1
            elif len(bad) < 5:
                bad.append((doc, s, e, lab, txt, t[s:e + 1]))
    conv = "inclusive" if inc > exc else "exclusive"
    return conv, dict(total_with_text=total, exclusive_match=exc, inclusive_match=inc, examples_no_match=bad)


def to_exclusive(gold, conv):
    """-> {doc: [(s, e_exclusive, label)]} (internal convention: python slice)"""
    add = 1 if conv == "inclusive" else 0
    return {d: [(s, e + add, lab) for s, e, lab, _ in ents] for d, ents in gold.items()}


# ======================================================================================================
# 2. Tokens and words with character offsets
# ======================================================================================================
# The pre-tokenisation decides where an entity can start or end. "bpe" = the split used in pretraining
# (a word keeps its punctuation: "Singh," is one piece). "fine" = punctuation is split off words, so
# "Singh" | "," can be cut exactly. Every piece is still encoded by the same BPE (same vocabulary).
SPLITS = {
    ("space", "bpe"): re.compile(r" ?\S+|\s+(?!\S)|\s+"),
    ("space", "fine"): re.compile(r" ?[^\W_]+| ?_+| ?[^\w\s]|\s+(?!\S)|\s+"),
    ("a1", "bpe"): re.compile(r"\S+|\s+"),
    ("a1", "fine"): re.compile(r"[^\W_]+|_+|[^\w\s]|\s+"),
}


def tokenizer_kind(tok):
    return "space" if getattr(tok, "space_prefix", False) else "a1"


def encode_words(tok, text, split="fine"):
    """text -> ids, wtok, wcs, wce
    ids  : BPE token ids of the whole text (decode(ids) == text)
    words: every pre-token piece that is not pure whitespace. For word w:
           wtok[w] = index of its first non-whitespace token (the token that represents the word),
           [wcs[w], wce[w]) = its characters without surrounding whitespace."""
    pat = SPLITS[(tokenizer_kind(tok), split)]
    ids, wtok, wcs, wce = [], [], [], []
    covered = 0
    for m in pat.finditer(text):
        chunk, base = m.group(), m.start()
        assert base == covered, f"pre-tokeniser skipped characters at {covered}"
        covered = m.end()
        char_of_byte = []
        for ci, ch in enumerate(chunk):
            char_of_byte.extend([ci] * len(ch.encode("utf-8")))
        pos, first, ws, we = 0, None, None, None
        for t in tok.encode_chunk(chunk.encode("utf-8")):
            n = len(tok.itos[t])
            s, e = base + char_of_byte[pos], base + char_of_byte[pos + n - 1] + 1
            pos += n
            while s < e and text[s].isspace():
                s += 1
            while e > s and text[e - 1].isspace():
                e -= 1
            if s < e:
                if first is None:
                    first, ws = len(ids), s
                we = e
            ids.append(t)
        if first is not None:
            wtok.append(first); wcs.append(ws); wce.append(we)
    assert covered == len(text), "pre-tokeniser did not cover the whole text"
    return ids, wtok, wcs, wce


# ======================================================================================================
# 3. Targets: BIO tags per word, and (start word, end word, label) for the span head
# ======================================================================================================
IGNORE = -100


def tag_vocab(labels):
    tags = ["O"] + [f"{p}-{l}" for l in labels for p in ("B", "I")]
    return tags, {t: i for i, t in enumerate(tags)}


def word_targets(wcs, wce, spans, tag2id, label2id):
    """gold char spans -> (BIO tag per word, [(first_word, last_word, label_id)], #overlapping spans skipped)"""
    n = len(wcs)
    tags, taken, wspans, conflicts = [0] * n, [False] * n, [], 0
    for s, e, lab in sorted(spans, key=lambda x: (x[0], -(x[1] - x[0]))):
        lo = bisect.bisect_right(wce, s)              # first word ending after s
        hi = bisect.bisect_left(wcs, e, lo)           # first word starting at/after e
        if lo >= hi:
            continue
        if any(taken[lo:hi]):
            conflicts += 1
            continue
        for k in range(lo, hi):
            tags[k] = tag2id[("B-" if k == lo else "I-") + lab]
            taken[k] = True
        wspans.append((lo, hi - 1, label2id[lab]))
    return tags, wspans, conflicts


def allowed_transitions(tags):
    """BIO validity: allowed[i][j] = may tag j follow tag i; start_ok[j] = may a sequence start with j."""
    K = len(tags)
    ok = [[True] * K for _ in range(K)]
    start_ok = [not t.startswith("I-") for t in tags]
    for i, a in enumerate(tags):
        for j, b in enumerate(tags):
            if b.startswith("I-") and not (a != "O" and a[2:] == b[2:]):
                ok[i][j] = False                      # I-X only after B-X or I-X
    return ok, start_ok


# ======================================================================================================
# 4. Decoding
# ======================================================================================================
def spans_from_tags(tags, wcs, wce, id2tag):
    """word tags -> [(s, e, label)]. B-X starts a span; I-X continues a span of type X; an I-X after O or
    after another type also starts a span (lenient repair, used with greedy argmax decoding)."""
    out, cur = [], None
    for t, s, e in zip(tags, wcs, wce):
        tag = id2tag[t]
        if tag == "O":
            if cur: out.append(cur); cur = None
            continue
        p, lab = tag.split("-", 1)
        if p == "B" or cur is None or cur[2] != lab:
            if cur: out.append(cur)
            cur = [s, e, lab]
        else:
            cur[1] = e
    if cur: out.append(cur)
    return [tuple(x) for x in out]


def spans_from_candidates(cands, wcs, wce, id2label):
    """span head: cands = {(i, j): (prob, label_id)} (label_id 0 = none) -> non-overlapping spans,
    highest probability first (greedy)."""
    used, out = set(), []
    for (i, j), (p, l) in sorted(cands.items(), key=lambda kv: -kv[1][0]):
        if l == 0 or any(k in used for k in range(i, j + 1)):
            continue
        used.update(range(i, j + 1))
        out.append((wcs[i], wce[j], id2label[l]))
    return sorted(out)


_WORDCH = re.compile(r"\w")


def propagate(spans, text, min_chars=4):
    """Document-level consistency: a string predicted as label L somewhere in the document is also
    labelled L at its other (unlabelled, whole-word) occurrences. Majority label per string."""
    votes = defaultdict(Counter)
    for s, e, l in spans:
        t = text[s:e]
        if len(t) >= min_chars and _WORDCH.search(t):
            votes[t][l] += 1
    covered = sorted((s, e) for s, e, _ in spans)
    starts = [s for s, _ in covered]

    def free(s, e):
        k = bisect.bisect_left(starts, e)
        return not any(cs < e and ce > s for cs, ce in covered[max(0, k - 50):k])

    out = list(spans)
    for t, c in votes.items():
        lab = c.most_common(1)[0][0]
        i = text.find(t)
        while i >= 0:
            j = i + len(t)
            boundary = (i == 0 or not text[i - 1].isalnum()) and (j == len(text) or not text[j].isalnum())
            if boundary and free(i, j):
                out.append((i, j, lab))
                k = bisect.bisect_left(starts, i); covered.insert(k, (i, j)); starts.insert(k, i)
            i = text.find(t, i + 1)
    return sorted(set(out))


def windows(n, size, stride):
    """window start positions covering [0, n); the last one is pulled back to end at n, so every window
    (except in documents shorter than size) has exactly `size` tokens - no padding needed."""
    if n <= size:
        return [0]
    starts = list(range(0, n - size + 1, stride))
    if starts[-1] + size < n:
        starts.append(n - size)
    return starts


# ======================================================================================================
# 5. Strict scoring and error analysis
# ======================================================================================================
def strict_scores(pred, gold, labels):
    """pred/gold: {doc: [(s,e,label)]} -> macro/micro P/R/F1 (exact start, end and label) + per label."""
    tp, fp, fn = Counter(), Counter(), Counter()
    for doc in set(gold) | set(pred):
        g = set(map(tuple, gold.get(doc, []))); p = set(map(tuple, pred.get(doc, [])))
        for x in p & g: tp[x[2]] += 1
        for x in p - g: fp[x[2]] += 1
        for x in g - p: fn[x[2]] += 1
    rows = {}
    for l in labels:
        P = tp[l] / (tp[l] + fp[l]) if tp[l] + fp[l] else 0.0
        R = tp[l] / (tp[l] + fn[l]) if tp[l] + fn[l] else 0.0
        F = 2 * P * R / (P + R) if P + R else 0.0
        rows[l] = dict(P=P, R=R, F1=F, tp=tp[l], fp=fp[l], fn=fn[l], support=tp[l] + fn[l])
    used = [l for l in labels if rows[l]["support"] or rows[l]["fp"]]
    mac = lambda k: sum(rows[l][k] for l in used) / max(1, len(used))
    T, FP, FN = sum(tp.values()), sum(fp.values()), sum(fn.values())
    micro = 2 * T / (2 * T + FP + FN) if T else 0.0
    return dict(macro_F1=mac("F1"), macro_P=mac("P"), macro_R=mac("R"), micro_F1=micro, per_label=rows)


def error_breakdown(pred, gold):
    """every mistake classified: wrong label on an exact span, boundary error (overlap, same label),
    boundary+label error, missed entity (no overlapping prediction), spurious prediction."""
    c = Counter()
    for doc in set(gold) | set(pred):
        g = set(map(tuple, gold.get(doc, []))); p = set(map(tuple, pred.get(doc, [])))
        c["correct"] += len(p & g)
        ps, used = sorted(p - g), set()
        for s, e, l in sorted(g - p):
            over = [x for x in ps if x[0] < e and x[1] > s]
            if not over:
                c["missed"] += 1; continue
            used.update(over)
            if any(o[0] == s and o[1] == e for o in over): c["wrong_label_exact_span"] += 1
            elif any(o[2] == l for o in over): c["boundary_error_same_label"] += 1
            else: c["boundary_and_label_error"] += 1
        c["spurious"] += len([x for x in ps if x not in used])
    return dict(c)


def label_confusion(pred, gold):
    """exact-span matches with a different label: Counter[(gold label, predicted label)]"""
    c = Counter()
    for doc in gold:
        pm = {(s, e): l for s, e, l in pred.get(doc, [])}
        for s, e, l in gold[doc]:
            if (s, e) in pm and pm[(s, e)] != l:
                c[(l, pm[(s, e)])] += 1
    return c


# ======================================================================================================
# 6. Submission
# ======================================================================================================
def to_submission(pred, conv, texts=None):
    """{doc: [(s, e_exclusive, label)]} -> {doc: [[s, e, label(, text)]]} in the dataset's own end convention"""
    sub = 1 if conv == "inclusive" else 0
    out = {}
    for doc, spans in pred.items():
        rows = []
        for s, e, l in sorted(spans):
            row = [int(s), int(e - sub), l]
            if texts is not None: row.append(texts[doc][s:e])
            rows.append(row)
        out[doc] = rows
    return out

In [ ]:
%%writefile p5src/ner_model.py
# Phase 2 (L-NER) model side:
#   Phase-1 backbones as feature extractors (all 4 architectures, bidirectional or causal),
#   fine-tuning / parameter-efficient methods (full, partial, BitFit, LoRA, LoRMA, Houlsby/Pfeiffer adapters,
#   linear probe), prediction heads (linear softmax, CRF, span classifier), knowledge distillation,
#   sliding-window training and document-level prediction.
import bisect
import math
import os
import pickle
import random
import sys
import time

import torch
import torch.nn as nn
import torch.nn.functional as F

import ner_data as nd

ARCH_INFO = {   # folder -> (tokenizer file, longest sequence the backbone supports)
    "arch1_decoder": ("a1_bpe8000.pkl", 1024),
    "arch2_encdec": ("a1_bpe8000.pkl", 768),
    "arch3_prefixlm": ("space_bpe8000.pkl", 1024),
    "arch4_mla": ("space_bpe8000.pkl", 1024),
}
NEG = -1e4


def autocast():
    return torch.autocast(device_type="cuda", dtype=torch.float16, enabled=torch.cuda.is_available())


def _add_code_path(root):
    code = os.path.join(root, "code")
    if code not in sys.path:
        sys.path.insert(0, code)


def load_tokenizer(arch, root):
    _add_code_path(root)
    import a1_tokenizer, space_bpe  # noqa: F401  (classes needed to unpickle)
    return pickle.load(open(os.path.join(root, "tokenizers", ARCH_INFO[arch][0]), "rb"))


# ======================================================================================================
# 1. Backbones
# ======================================================================================================
def _mla_forward(self, x, cos, sin, cache=None):
    """MLA attention of architecture 4 without the causal mask (used when self.bidirectional is True)."""
    if cache is not None or not getattr(self, "bidirectional", False):
        return self._causal_forward(x, cos, sin, cache)
    from rope import apply_rope
    B, T, _ = x.shape
    q = self.q_proj(x).view(B, T, self.h, self.nope + self.rope).transpose(1, 2)
    q_nope, q_rope = q.split([self.nope, self.rope], dim=-1)
    q_rope = apply_rope(q_rope, cos.to(q.dtype), sin.to(q.dtype))
    c_kv = self.kv_norm(self.kv_down(x))
    k_r = apply_rope(self.k_rope(x)[:, None], cos.to(q.dtype), sin.to(q.dtype))
    k_nope = self.k_up(c_kv).view(B, T, self.h, self.nope).transpose(1, 2)
    v = self.v_up(c_kv).view(B, T, self.h, self.vd).transpose(1, 2)
    q = torch.cat([q_nope, q_rope], dim=-1)
    k = torch.cat([k_nope, k_r.expand(B, self.h, T, self.rope)], dim=-1)
    out = F.scaled_dot_product_attention(q, k, v, is_causal=False, dropout_p=self.dropout if self.training else 0.0)
    return self.o_proj(out.transpose(1, 2).reshape(B, T, -1)), None


class Backbone(nn.Module):
    """Phase-1 language model reduced to a feature extractor: token ids (B,T) -> hidden states (B,T,d).
    bidirectional: drop the causal mask (arch 1, 4), use the whole window as prefix (arch 3); the arch-2
    encoder is bidirectional by design. keep_layers: keep only the first k blocks (distillation student).
    init="random": same architecture, random weights (control experiment for transfer from Phase 1)."""

    def __init__(self, arch, root, bidirectional=True, keep_layers=None, init="pretrained"):
        super().__init__()
        _add_code_path(root)
        self.arch, self.bidirectional = arch, bidirectional
        ck = torch.load(os.path.join(root, arch, "ckpt_best.pt"), map_location="cpu", weights_only=False)
        cfg = dict(ck["cfg"])
        if arch == "arch4_mla":
            from mla import MLAConfig, MLALM, MLAttention
            if not hasattr(MLAttention, "_causal_forward"):
                MLAttention._causal_forward = MLAttention.forward
                MLAttention.forward = _mla_forward
            m = MLALM(MLAConfig(**cfg))
        elif arch == "arch3_prefixlm":
            from prefixlm import PrefixLMConfig, PrefixLM
            m = PrefixLM(PrefixLMConfig(**cfg))
        elif arch == "arch2_encdec":
            from encdec import EncDecConfig, EncDecLM
            m = EncDecLM(EncDecConfig(**cfg))
        elif arch == "arch1_decoder":
            from model_arch import LMConfig, DecoderLM
            m = DecoderLM(LMConfig(**cfg))
        else:
            raise ValueError(arch)
        if init == "pretrained":
            missing, unexpected = m.load_state_dict(ck["model"], strict=False)
            assert not unexpected and all(x in ("rope", "pos_emb", "mask") or "rope" in x or "mask" in x
                                          or "pos_emb" in x for x in missing), (missing[:5], unexpected[:5])
        else:
            for mod in m.modules():
                if isinstance(mod, (nn.Linear, nn.Embedding)):
                    nn.init.normal_(mod.weight, std=0.02)
                    if getattr(mod, "bias", None) is not None: nn.init.zeros_(mod.bias)
        self.phase1_step, self.phase1_val = ck.get("step"), ck.get("val_loss")
        if arch == "arch2_encdec":                        # NER uses only the (bidirectional) encoder
            m.decoder = nn.ModuleList(); m.dec_norm = nn.Identity()
        if arch == "arch1_decoder":                       # A3 attention keeps its causal mask in a buffer
            for blk in m.blocks:
                blk.attn.causal_mask.fill_(True) if bidirectional else None
        if arch == "arch4_mla":
            for blk in m.blocks:
                blk.attn.bidirectional = bidirectional
        if keep_layers:
            blocks = self._blocks(m)
            del blocks[keep_layers:]
        m.head = None if arch != "arch2_encdec" else m.head   # LM head is not used (tied to the embedding)
        self.m, self.d, self.max_len = m, cfg["d_model"], ARCH_INFO[arch][1]

    @staticmethod
    def _blocks(m):
        return m.encoder if hasattr(m, "encoder") else m.blocks

    @property
    def blocks(self):
        return self._blocks(self.m)

    @property
    def final_norm(self):
        m = self.m
        return m.enc_norm if hasattr(m, "enc_norm") else (m.ln_f if hasattr(m, "ln_f") else m.norm)

    def forward(self, ids):
        m = self.m
        if self.arch == "arch4_mla":
            return m.hidden(ids)
        if self.arch == "arch3_prefixlm":
            return m.hidden(ids, prefix_len=ids.size(1) if self.bidirectional else 0)
        if self.arch == "arch2_encdec":
            return m.encode(ids)
        B, T = ids.shape                                  # arch 1: DecoderLM.forward without the LM head
        x = m.tok_emb(ids)
        if m.cfg.scale_emb:
            x = x * math.sqrt(m.cfg.d_model)
        if m.cfg.pos_emb_type == "sinusoidal":
            x = x + m.pos_emb[:T].unsqueeze(0)
        for blk in m.blocks:
            x = blk(x)
        return m.ln_f(x)


# ======================================================================================================
# 2. Fine-tuning / adaptation methods
# ======================================================================================================
class LoRALinear(nn.Module):
    """LoRA (Hu et al. 2021): W x + (alpha/r) * B A x, with W frozen, A random, B = 0 (starts at W)."""

    def __init__(self, base, r=16, alpha=32, dropout=0.05):
        super().__init__()
        self.base, self.scale = base, alpha / r
        self.A = nn.Parameter(torch.empty(r, base.in_features)); nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        return self.base(x) + self.scale * F.linear(F.linear(self.drop(x), self.A), self.B)


class LoRMALinear(nn.Module):
    """LoRMA (low-rank MULTIPLICATIVE adaptation): W' = (I + (alpha/r) B A) W, with W frozen.
    The identity term gives the product full rank ("rank inflation" by identity), and B = 0 makes the
    adapted layer start exactly at the pretrained one. Computed as h + s * B(A h) with h = W x, so the
    d_out x d_in product (BA)W is never formed. A bias, if any, is added after the multiplicative update."""

    def __init__(self, base, r=16, alpha=32):
        super().__init__()
        self.base, self.scale = base, alpha / r
        self.A = nn.Parameter(torch.empty(r, base.out_features)); nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))

    def forward(self, x):
        h = F.linear(x, self.base.weight)
        h = h + self.scale * F.linear(F.linear(h, self.A), self.B)
        return h if self.base.bias is None else h + self.base.bias


class Adapter(nn.Module):
    """bottleneck adapter (Houlsby et al. 2019): x + up(GELU(down(x))), up = 0 at start (identity)."""

    def __init__(self, d, bottleneck=64):
        super().__init__()
        self.down, self.up = nn.Linear(d, bottleneck), nn.Linear(bottleneck, d)
        nn.init.normal_(self.down.weight, std=0.02); nn.init.zeros_(self.down.bias)
        nn.init.zeros_(self.up.weight); nn.init.zeros_(self.up.bias)

    def forward(self, x):
        return x + self.up(F.gelu(self.down(x)))


class AdapterWrap(nn.Module):
    """applies an adapter to the output of a sub-layer (attention or feed-forward), before the residual add"""

    def __init__(self, module, d, bottleneck):
        super().__init__()
        self.module, self.adapter = module, Adapter(d, bottleneck)

    def __getattr__(self, name):                      # keep attributes of the wrapped module reachable
        try:
            return super().__getattr__(name)
        except AttributeError:
            return getattr(self._modules["module"], name)

    def forward(self, *args, **kwargs):
        out = self.module(*args, **kwargs)
        if isinstance(out, tuple):
            return (self.adapter(out[0]),) + tuple(out[1:])
        return self.adapter(out)


def _replace_linears(root_module, make):
    n = 0
    for name, child in list(root_module.named_children()):
        if isinstance(child, nn.Linear):
            setattr(root_module, name, make(child)); n += 1
        else:
            n += _replace_linears(child, make)
    return n


def apply_method(net, cfg):
    """Freeze/unfreeze and insert adaptation modules according to cfg["method"]. The task head is always
    trained. Returns a short description for the record."""
    bb, method = net.backbone, cfg["method"]
    for p in bb.parameters():
        p.requires_grad = method == "full_ft"
    desc = method
    if method == "partial":                           # layer-wise: only the top k blocks + final norm
        k = cfg.get("train_top_layers", 4)
        for blk in list(bb.blocks)[-k:]:
            for p in blk.parameters(): p.requires_grad = True
        for p in bb.final_norm.parameters(): p.requires_grad = True
        desc = f"partial (top {k} of {len(bb.blocks)} blocks)"
    elif method == "bitfit":                          # only bias terms (Ben Zaken et al. 2022)
        n = 0
        for name, p in bb.named_parameters():
            if name.endswith(".bias"):
                p.requires_grad = True; n += p.numel()
        assert n > 0, "BitFit needs bias terms - this backbone has none (use arch 1 or arch 2)"
        desc = f"bitfit ({n:,} bias parameters)"
    elif method in ("lora", "lorma"):
        r, a = cfg.get("rank", 16), cfg.get("alpha", 32)
        make = (lambda lin: LoRALinear(lin, r, a, cfg.get("lora_dropout", 0.05))) if method == "lora" \
            else (lambda lin: LoRMALinear(lin, r, a))
        n = sum(_replace_linears(blk, make) for blk in bb.blocks)
        for mod in bb.modules():
            if isinstance(mod, (LoRALinear, LoRMALinear)):
                mod.A.requires_grad = True; mod.B.requires_grad = True
        desc = f"{method} (rank {r}, alpha {a}, {n} linear layers)"
    elif method in ("houlsby", "pfeiffer"):           # Houlsby: after attention and FFN; Pfeiffer: after FFN only
        bott = cfg.get("bottleneck", 64)
        for blk in bb.blocks:
            if method == "houlsby":
                blk.attn = AdapterWrap(blk.attn, bb.d, bott)
            blk.ffn = AdapterWrap(blk.ffn, bb.d, bott)
        for mod in bb.modules():
            if isinstance(mod, Adapter):
                for p in mod.parameters(): p.requires_grad = True
            if isinstance(mod, (nn.LayerNorm,)) or type(mod).__name__ == "RMSNorm":
                for p in mod.parameters(): p.requires_grad = True     # Houlsby also trains the norms
        desc = f"{method} adapters (bottleneck {bott})"
    elif method == "probe":                           # frozen backbone, only the head (linear probe)
        desc = "linear probe (frozen backbone)"
    elif method != "full_ft":
        raise ValueError(method)
    return desc


def count_params(net):
    total = sum(p.numel() for p in net.parameters())
    train = sum(p.numel() for p in net.parameters() if p.requires_grad)
    return total, train


# ======================================================================================================
# 3. Prediction heads
# ======================================================================================================
class CRF(nn.Module):
    """linear-chain CRF over word tags. Learned transitions + fixed BIO constraints (forbidden moves = -1e4)."""

    def __init__(self, tags):
        super().__init__()
        K = len(tags)
        ok, start_ok = nd.allowed_transitions(tags)
        self.trans = nn.Parameter(torch.zeros(K, K)); self.start = nn.Parameter(torch.zeros(K))
        self.end = nn.Parameter(torch.zeros(K))
        self.register_buffer("ban", (~torch.tensor(ok)).float() * NEG)
        self.register_buffer("ban_start", (~torch.tensor(start_ok)).float() * NEG)

    def scores(self, train=False):
        # training windows may start inside an entity (first word I-X), so the start constraint is only
        # applied when decoding whole documents
        return self.trans + self.ban, (self.start if train else self.start + self.ban_start), self.end

    def nll(self, em, tags, mask):
        """em (B,W,K) float, tags (B,W) (pads ignored), mask (B,W) bool, prefix-contiguous"""
        trans, start, end = self.scores(train=True)
        B, W, K = em.shape
        tags = tags.clamp(min=0)
        mf = mask.float()
        # score of the gold path
        gold = start[tags[:, 0]] + em[:, 0].gather(1, tags[:, :1]).squeeze(1)
        for t in range(1, W):
            step = trans[tags[:, t - 1], tags[:, t]] + em[:, t].gather(1, tags[:, t:t + 1]).squeeze(1)
            gold = gold + step * mf[:, t]
        last = tags.gather(1, (mask.sum(1) - 1).clamp(min=0)[:, None]).squeeze(1)
        gold = gold + end[last]
        # log partition
        alpha = start[None] + em[:, 0]
        for t in range(1, W):
            nxt = torch.logsumexp(alpha[:, :, None] + trans[None], dim=1) + em[:, t]
            alpha = torch.where(mask[:, t, None], nxt, alpha)
        logz = torch.logsumexp(alpha + end[None], dim=1)
        has = mask[:, 0]
        return ((logz - gold) * has.float()).sum() / has.float().sum().clamp(min=1)


def viterbi(em, trans, start, end):
    """em (n,K) -> best tag sequence (list) under transition scores (CPU)"""
    n, K = em.shape
    if n == 0:
        return []
    score = start + em[0]
    back = []
    for t in range(1, n):
        s = score[:, None] + trans
        best, idx = s.max(0)
        score = best + em[t]; back.append(idx)
    score = score + end
    tag = int(score.argmax())
    path = [tag]
    for idx in reversed(back):
        tag = int(idx[tag]); path.append(tag)
    return path[::-1]


class NERModel(nn.Module):
    """backbone -> word representations (first token of each word) -> head.
    head = "linear": softmax over BIO tags (cross-entropy, optional class weights)
           "crf":    BIO tag emissions + CRF (sequence-level likelihood)
           "span":   classify every word span up to max_span_words (start/end projections, bilinear term,
                     width embedding), label 0 = not an entity."""

    def __init__(self, backbone, tags, labels, head="linear", dropout=0.2, class_weights=None,
                 max_span_words=32, span_dim=256, neg_weight=0.5):
        super().__init__()
        self.backbone, self.head_type, self.tags = backbone, head, tags
        self.id2label = {i + 1: l for i, l in enumerate(labels)}          # span head: class 0 = no entity
        n_labels = len(labels)
        d, K = backbone.d, len(tags)
        self.drop = nn.Dropout(dropout)
        if head in ("linear", "crf"):
            self.head = nn.Linear(d, K)
            nn.init.normal_(self.head.weight, std=0.02); nn.init.zeros_(self.head.bias)
            self.register_buffer("class_weights", class_weights if class_weights is not None else torch.ones(K))
            if head == "crf":
                self.crf = CRF(tags)
        elif head == "span":
            C = n_labels + 1
            self.maxw = max_span_words
            self.head = nn.ModuleDict(dict(
                s=nn.Sequential(nn.Linear(d, span_dim), nn.GELU()), e=nn.Sequential(nn.Linear(d, span_dim), nn.GELU()),
                ls=nn.Linear(span_dim, C), le=nn.Linear(span_dim, C), bil=nn.Linear(span_dim, C, bias=False),
                width=nn.Embedding(max_span_words, C)))
            w = torch.ones(C); w[0] = neg_weight
            self.register_buffer("class_weights", w)
        else:
            raise ValueError(head)

    def word_states(self, ids, wpos):
        h = self.backbone(ids)                                            # (B,T,d)
        return h.gather(1, wpos[:, :, None].expand(-1, -1, h.size(-1)))   # (B,W,d)

    def span_logits(self, hw, wmask):
        H = self.head
        s, e = H["s"](hw), H["e"](hw)                                     # (B,W,p)
        B, W, p = s.shape
        k = torch.arange(self.maxw, device=hw.device)
        j = torch.arange(W, device=hw.device)[:, None] + k[None]          # (W,maxw)
        valid = (j < W)[None] & wmask[:, :, None] & wmask.gather(1, j.clamp(max=W - 1).reshape(1, -1).expand(B, -1)).view(B, W, -1)
        ej = e.gather(1, j.clamp(max=W - 1).reshape(1, -1, 1).expand(B, -1, p)).view(B, W, self.maxw, p)
        logits = H["ls"](s)[:, :, None] + H["le"](ej) + H["bil"](s[:, :, None] * ej) + H["width"](k)[None, None]
        return logits.float(), valid

    def forward(self, ids, wpos, wmask, wtags=None, stgt=None):
        with autocast():
            hw = self.drop(self.word_states(ids, wpos))
            if self.head_type == "span":
                logits, valid = self.span_logits(hw, wmask)
            else:
                logits = self.head(hw).float()
        if self.head_type == "span":
            if stgt is None:                                   # invalid slots -> certainly "no entity"
                inv = ~valid
                out = logits.masked_fill(inv[..., None], NEG)
                out[..., 0] = out[..., 0].masked_fill(inv, 0.0)
                return out
            tgt = stgt.masked_fill(~valid, nd.IGNORE)
            return F.cross_entropy(logits.reshape(-1, logits.size(-1)), tgt.reshape(-1),
                                   weight=self.class_weights, ignore_index=nd.IGNORE).reshape(1)
        if wtags is None:
            return logits
        if self.head_type == "crf":
            return self.crf.nll(logits, wtags, wmask).reshape(1)
        return F.cross_entropy(logits.reshape(-1, logits.size(-1)), wtags.reshape(-1),
                               weight=self.class_weights, ignore_index=nd.IGNORE).reshape(1)


class KDWrapper(nn.Module):
    """Knowledge distillation (Hinton et al. 2015) at word level:
    loss = a * CE(student, gold) + (1 - a) * T^2 * KL(teacher_T || student_T). The teacher is frozen."""

    def __init__(self, student, teacher, T=2.0, alpha=0.5):
        super().__init__()
        self.student, self.teacher, self.T, self.alpha = student, teacher, T, alpha
        for p in teacher.parameters(): p.requires_grad = False

    def train(self, mode=True):
        super().train(mode); self.teacher.eval(); return self

    def forward(self, ids, wpos, wmask, wtags=None, stgt=None):
        if wtags is None:
            return self.student(ids, wpos, wmask)
        with torch.no_grad():
            t = self.teacher(ids, wpos, wmask)                            # (B,W,K)
        s = self.student(ids, wpos, wmask)
        m = wmask.reshape(-1)
        ce = F.cross_entropy(s.reshape(-1, s.size(-1)), wtags.reshape(-1), ignore_index=nd.IGNORE)
        sl, tl = s.reshape(-1, s.size(-1))[m] / self.T, t.reshape(-1, t.size(-1))[m] / self.T
        kl = F.kl_div(F.log_softmax(sl, -1), F.softmax(tl, -1), reduction="batchmean") * self.T ** 2
        return (self.alpha * ce + (1 - self.alpha) * kl).reshape(1)


# ======================================================================================================
# 4. Documents -> windows -> batches
# ======================================================================================================
def prepare_docs(texts, gold, tok, tag2id, label2id, split="fine"):
    docs, conflicts = {}, 0
    for d, t in texts.items():
        ids, wtok, wcs, wce = nd.encode_words(tok, t, split)
        r = dict(ids=ids, wtok=wtok, wcs=wcs, wce=wce)
        if gold is not None:
            r["wtags"], r["wspans"], c = nd.word_targets(wcs, wce, gold.get(d, []), tag2id, label2id); conflicts += c
        docs[d] = r
    return docs, conflicts


class Windowing:
    """Cuts every document into windows of `size` tokens. With header_tokens > 0, the first tokens of the
    document (case title, parties, counsel) are put in front of every window, followed by a separator,
    so each window can see who the appellant/respondent/counsel are. Header words are never labelled."""

    def __init__(self, tok, size, header_tokens=0, max_len=1024):
        self.sep = tok.encode("\n\n") if header_tokens else []
        self.H = header_tokens
        self.size = min(size, max_len - (header_tokens + len(self.sep) if header_tokens else 0))
        assert self.size >= 64, "window too small"

    def windows(self, docs, stride):
        out = []
        for d, r in docs.items():
            if r["wtok"]:
                for s in nd.windows(len(r["ids"]), self.size, stride):
                    out.append((d, s, min(self.size, len(r["ids"]))))
        return out

    def window_words(self, r, s, L):
        w0 = bisect.bisect_left(r["wtok"], s); w1 = bisect.bisect_left(r["wtok"], s + L)
        return w0, w1

    def batch(self, docs, b, device, head=None, maxw=None):
        ids, wpos, wtags, ranges, stgts = [], [], [], [], []
        for d, s, L in b:
            r = docs[d]
            head_ids = (r["ids"][:self.H] + self.sep) if self.H else []
            off = len(head_ids) - s
            ids.append(head_ids + r["ids"][s:s + L])
            w0, w1 = self.window_words(r, s, L)
            ranges.append((w0, w1))
            wpos.append([r["wtok"][w] + off for w in range(w0, w1)])
            if "wtags" in r:
                wtags.append(r["wtags"][w0:w1])
        T = len(ids[0]); assert all(len(x) == T for x in ids)
        Wm = max(1, max(len(x) for x in wpos))
        P = torch.zeros(len(b), Wm, dtype=torch.long); M = torch.zeros(len(b), Wm, dtype=torch.bool)
        for i, x in enumerate(wpos):
            P[i, :len(x)] = torch.tensor(x, dtype=torch.long); M[i, :len(x)] = True
        out = dict(ids=torch.tensor(ids, dtype=torch.long), wpos=P, wmask=M)
        if wtags:
            Tg = torch.full((len(b), Wm), nd.IGNORE, dtype=torch.long)
            for i, x in enumerate(wtags):
                Tg[i, :len(x)] = torch.tensor(x, dtype=torch.long)
            out["wtags"] = Tg
            if head == "span":
                S = torch.zeros(len(b), Wm, maxw, dtype=torch.long)
                for i, ((d, s, L), (w0, w1)) in enumerate(zip(b, ranges)):
                    for gi, gj, l in docs[d]["wspans"]:
                        if gj < w0 or gi >= w1:
                            continue
                        if gi >= w0 and gj < w1:                    # entity inside the window
                            if gj - gi < maxw: S[i, gi - w0, gj - gi] = l
                        elif gi >= w0:                              # runs past the window end: unknown
                            S[i, gi - w0, :] = nd.IGNORE
                        else:                                       # started before the window: unknown
                            for a in range(max(w0, gj - maxw + 1), gj + 1):
                                S[i, a - w0, gj - a] = nd.IGNORE
                out["stgt"] = S
        return {k: v.to(device) for k, v in out.items()}, ranges


def length_batches(win, bs, shuffle, rng=None):
    by_len = {}
    for w in win:
        by_len.setdefault(w[2], []).append(w)
    out = []
    for _, ws in sorted(by_len.items()):
        if shuffle: rng.shuffle(ws)
        out += [ws[i:i + bs] for i in range(0, len(ws), bs)]
    if shuffle: rng.shuffle(out)
    return out


# ======================================================================================================
# 5. Prediction: windows -> document-level scores -> spans
# ======================================================================================================
@torch.no_grad()
def predict_scores(net, docs, wd, stride, bs, device):
    """returns per document: linear -> averaged tag probabilities (n_words,K); crf -> averaged emissions;
    span -> {(i,j): averaged class probabilities}. Overlapping windows are combined with a tent weight
    (a word in the middle of a window has more context than one at its edge)."""
    net.eval()
    model = net.student if isinstance(net, KDWrapper) else net
    head = model.head_type
    out = {}
    for b in length_batches(wd.windows(docs, stride), bs, shuffle=False):
        batch, ranges = wd.batch(docs, b, device)
        res = net(batch["ids"], batch["wpos"], batch["wmask"]).float().cpu()
        for k, ((d, s, L), (w0, w1)) in enumerate(zip(b, ranges)):
            n = w1 - w0
            if n == 0:
                continue
            r = docs[d]
            pos = torch.tensor([r["wtok"][w] - s for w in range(w0, w1)], dtype=torch.float)
            wt = 1 + torch.minimum(pos, L - 1 - pos)
            if head == "span":
                cand = out.setdefault(d, {})
                pr = res[k, :n].softmax(-1)                                 # (n, maxw, C)
                keep = (pr[..., 1:].max(-1).values > 0.2).nonzero().tolist()     # plausible entities only
                for i, kk in keep:
                    j = i + kk
                    if j >= n: continue
                    key = (w0 + i, w0 + j); ww = float(min(wt[i], wt[j]))
                    acc = cand.get(key)
                    cand[key] = [pr[i, kk] * ww, ww] if acc is None else [acc[0] + pr[i, kk] * ww, acc[1] + ww]
            else:
                if d not in out:
                    nw = len(r["wtok"]); out[d] = [torch.zeros(nw, res.size(-1)), torch.zeros(nw, 1)]
                v = res[k, :n].softmax(-1) if head == "linear" else res[k, :n]
                out[d][0][w0:w1] += v * wt[:, None]; out[d][1][w0:w1] += wt[:, None]
    final = {}
    for d in docs:
        if head == "span":
            final[d] = {key: a / w for key, (a, w) in out.get(d, {}).items()}
        else:
            v = out.get(d)
            final[d] = (v[0] / v[1].clamp(min=1e-9)) if v is not None else torch.zeros(0, len(model.tags))
    return final


def decode(scores, docs, model, texts, strategy="viterbi", propagate=False):
    """document scores -> {doc: [(s, e, label)]}.
    linear head: "greedy" (argmax per word + lenient BIO repair) or "viterbi" (best BIO-valid sequence
    under the word log-probabilities); crf head: Viterbi with the learned transitions; span head: greedy
    non-overlapping spans by probability. propagate: document-level label consistency (see ner_data)."""
    head, tags = model.head_type, model.tags
    id2tag = dict(enumerate(tags))
    pred = {}
    if head != "span":
        if head == "crf":
            trans, start, end = [x.detach().float().cpu() for x in model.crf.scores()]
        else:
            ok, start_ok = nd.allowed_transitions(tags)
            trans = (~torch.tensor(ok)).float() * NEG; start = (~torch.tensor(start_ok)).float() * NEG
            end = torch.zeros(len(tags))
    for d, sc in scores.items():
        r = docs[d]
        if head == "span":
            cands = {k: (float(v[1:].max()), int(v.argmax())) for k, v in sc.items()}
            spans = nd.spans_from_candidates(cands, r["wcs"], r["wce"], model.id2label)
        elif head == "linear" and strategy == "greedy":
            spans = nd.spans_from_tags(sc.argmax(-1).tolist(), r["wcs"], r["wce"], id2tag)
        else:
            em = sc.clamp(min=1e-9).log() if head == "linear" else sc
            spans = nd.spans_from_tags(viterbi(em, trans, start, end), r["wcs"], r["wce"], id2tag)
        pred[d] = nd.propagate(spans, texts[d]) if propagate else spans
    return pred


def decoding_options(head):
    base = ["greedy", "viterbi"] if head == "linear" else ["viterbi"] if head == "crf" else ["greedy"]
    return [(s, p) for s in base for p in (False, True)]


# ======================================================================================================
# 6. Training
# ======================================================================================================
def train(net, train_docs, val_docs, val_texts, val_gold, labels, wd, cfg, device, log=print):
    """AdamW (backbone/adaptation LR + head LR), linear warm-up and decay, fp16, gradient accumulation,
    DataParallel over all GPUs; evaluates strict macro-F1 on the validation documents after every epoch
    and keeps the best weights (early stopping with patience)."""
    rng = random.Random(cfg["seed"])
    model = net.student if isinstance(net, KDWrapper) else net
    gpus = torch.cuda.device_count()
    para = nn.DataParallel(net) if gpus > 1 else net
    is_head = lambda n: n.startswith(("head.", "crf.", "student.head.", "student.crf."))
    named = [(n, p) for n, p in net.named_parameters() if p.requires_grad]
    groups = [dict(params=[p for n, p in named if not is_head(n)], lr=cfg["lr"]),
              dict(params=[p for n, p in named if is_head(n)], lr=cfg["lr_head"])]
    groups = [g for g in groups if g["params"]]
    for g in groups: g["base_lr"] = g["lr"]
    params = [p for g in groups for p in g["params"]]
    opt = torch.optim.AdamW(groups, weight_decay=cfg["weight_decay"])
    scaler = torch.amp.GradScaler("cuda", enabled=torch.cuda.is_available())
    win = wd.windows(train_docs, cfg["train_stride"])
    acc = cfg.get("accum", 1)
    steps_per_epoch = math.ceil(len(win) / cfg["batch_size"] / acc)
    total = steps_per_epoch * cfg["epochs"]; warm = max(1, int(cfg["warmup_frac"] * total))
    log(f"train windows {len(win):,} of {wd.size} tokens (+{wd.H} header) | {steps_per_epoch} steps/epoch x "
        f"{cfg['epochs']} epochs | trainable {sum(p.numel() for p in params):,} | GPUs {max(gpus, 1)}")
    best, best_state, history, step, bad, t0 = -1.0, None, [], 0, 0, time.time()
    if torch.cuda.is_available(): torch.cuda.reset_peak_memory_stats()
    maxw = getattr(model, "maxw", None)
    for ep in range(1, cfg["epochs"] + 1):
        net.train(); run, n = 0.0, 0
        bl = length_batches(win, cfg["batch_size"], shuffle=True, rng=rng)
        for bi, b in enumerate(bl):
            batch, _ = wd.batch(train_docs, b, device, model.head_type, maxw)
            loss = para(**batch).mean() / acc
            scaler.scale(loss).backward()
            run += loss.item() * acc; n += 1
            if (bi + 1) % acc and bi + 1 < len(bl):
                continue
            step += 1
            f = step / warm if step <= warm else max(0.0, (total - step) / max(1, total - warm))
            for g in opt.param_groups: g["lr"] = g["base_lr"] * f
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(params, cfg["clip"])
            scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True)
            if step % cfg["log_every"] == 0:
                log(f"  ep {ep} step {step}/{total} | loss {run / n:.4f} | {(time.time() - t0) / 60:.1f} min")
                run, n = 0.0, 0
        if cfg.get("select_last"):                    # refit on all documents: fixed epochs, no selection
            history.append(dict(epoch=ep, step=step, minutes=round((time.time() - t0) / 60, 1)))
            log(f"epoch {ep} done (refit on all documents: no validation-based selection)")
            continue
        sc = predict_scores(net, val_docs, wd, cfg["eval_stride"], cfg["eval_batch"], device)
        strategy = "viterbi" if model.head_type != "span" else "greedy"
        res = nd.strict_scores(decode(sc, val_docs, model, val_texts, strategy), val_gold, labels)
        history.append(dict(epoch=ep, step=step, macro_F1=res["macro_F1"], micro_F1=res["micro_F1"],
                            macro_P=res["macro_P"], macro_R=res["macro_R"], minutes=round((time.time() - t0) / 60, 1)))
        note = ""
        if res["macro_F1"] > best + 1e-4:
            best, bad, note = res["macro_F1"], 0, "  <- best"
            best_state = {k: v.detach().to("cpu", copy=True) for k, v in net.state_dict().items()}
        else:
            bad += 1
        log(f"epoch {ep}: val strict macro-F1 {res['macro_F1']:.4f} | micro {res['micro_F1']:.4f} | "
            f"P {res['macro_P']:.3f} R {res['macro_R']:.3f}{note}")
        if cfg.get("patience") and bad >= cfg["patience"]:
            log(f"early stopping: no improvement for {bad} epochs"); break
        if (time.time() - t0) / 3600 > cfg["max_train_hours"]:
            log("time budget reached - stopping"); break
    if best_state is not None:
        net.load_state_dict(best_state)
    peak = torch.cuda.max_memory_allocated() / 2**30 if torch.cuda.is_available() else 0.0
    return dict(best_val_macro_F1=best, history=history, train_minutes=round((time.time() - t0) / 60, 1),
                peak_gpu_gb=round(peak, 2))


def trainable_state(net):
    """only the trained tensors (what a PEFT method has to store per task)"""
    names = {n for n, p in net.named_parameters() if p.requires_grad}
    return {k: v.detach().cpu() for k, v in net.state_dict().items() if k in names}


@torch.no_grad()
def self_test(net, wd, docs, device, head, maxw=None):
    """one forward + backward on two real windows before the long run (catches shape/device errors early)"""
    b = wd.windows(docs, wd.size)[:2]
    if len(b) == 2 and b[0][2] != b[1][2]: b = b[:1]
    batch, _ = wd.batch(docs, b, device, head, maxw)
    net.train()
    with torch.enable_grad():
        loss = net(**batch).mean(); loss.backward()
    net.zero_grad(set_to_none=True)
    out = net(batch["ids"], batch["wpos"], batch["wmask"])
    return float(loss), tuple(out.shape)

In [ ]:
%%writefile p5src/ls.py
# Phase 5 (Legal Statute Identification, multi-label) - hierarchical document encoder on a Phase-1 backbone.
#   case text -> Phase-1 tokenizer -> up to K windows covering the WHOLE document (consecutive if it fits, otherwise
#   K evenly spaced windows) -> Phase-1 backbone (+ full FT / LoRMA / partial FT ...) -> per-window learned attention
#   pooling over tokens (+ mean) -> window-position embedding + BiGRU over windows + attention and MAX pooling
#   (a statute cue can sit in any window) -> one logit per statute (binary relevance).
#   Loss: BCE with per-statute positive weights (rare statutes), optional distillation from teacher probabilities.
#   Decision: per-statute thresholds tuned on dev (falling back to a global one for statutes with few dev positives),
#   at least one statute per document.  Phase-4 lessons: attention pooling, causal mask kept for arch 4 + LoRMA.
import json
import math
import os
import random
import time
import zipfile
from collections import Counter

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

import ner_model as nm

VERSION = "p5-v3"                                     # v3: + asymmetric loss option, bagged thresholds                                     # v2: interim probabilities + zip saved after every evaluation

# ======================================================================================================
# 1. Data
# ======================================================================================================
_ID = ("id", "case_id", "doc_id", "uid", "name")
_LAB = ("labels", "label", "statutes", "statute_ids", "targets", "y")


def read_json_any(path):
    with open(path, encoding="utf-8") as f:
        head = f.read(1)
    if head in "[{":
        try:
            return json.load(open(path, encoding="utf-8"))
        except json.JSONDecodeError:
            pass
    return [json.loads(l) for l in open(path, encoding="utf-8") if l.strip()]


def _as_label(x):
    if isinstance(x, bool):
        return int(x)
    if isinstance(x, (int, np.integer)):
        return int(x)
    try:
        return int(str(x).strip())
    except ValueError:
        return str(x).strip()


def parse(raw):
    """any common shape -> list of dict(id=<original>, text=str, labels=[unique statute ids] or None)"""
    if isinstance(raw, dict):
        for k in ("data", "examples", "docs", "cases"):
            if k in raw and isinstance(raw[k], (list, dict)):
                raw = raw[k]; break
    if isinstance(raw, dict):
        raw = [dict(v, id=v.get("id", k)) if isinstance(v, dict) else dict(id=k, text=v) for k, v in raw.items()]
    out = []
    for i, d in enumerate(raw):
        did = next((d[k] for k in _ID if k in d), i)
        t = next((d[k] for k in ("text", "document", "facts", "case_text", "content") if k in d), "")
        if isinstance(t, list):
            t = " ".join(map(str, t))
        lab = next((d[k] for k in _LAB if k in d), None)
        if lab is not None and not isinstance(lab, (list, tuple)):
            lab = [lab]
        labs = None if lab is None else sorted({_as_label(x) for x in lab}, key=lambda v: (isinstance(v, str), v))
        out.append(dict(id=did, text=" ".join(str(t).split()), labels=labs))
    return out


_TOK = None


def _init_tok(arch, root):
    global _TOK
    _TOK = nm.load_tokenizer(arch, root)


def _encode(args):
    text, max_chars = args
    pre = " " if getattr(_TOK, "space_prefix", False) else ""
    return np.asarray(_TOK.encode(pre + text[:max_chars]), dtype=np.int32)


def tokenize(docs, arch, root, max_chars=40000, workers=4, log=print):
    t0 = time.time()
    args = [(d["text"], max_chars) for d in docs]
    if workers > 1:
        import multiprocessing as mp
        with mp.get_context("fork").Pool(workers, initializer=_init_tok, initargs=(arch, root)) as pool:
            enc = pool.map(_encode, args, chunksize=64)
    else:
        _init_tok(arch, root); enc = [_encode(a) for a in args]
    items = [dict(doc=d, ids=e) for d, e in zip(docs, enc)]
    n = np.array([len(e) for e in enc])
    log(f"tokenised {len(items):,} documents in {time.time() - t0:.0f}s | tokens: median {int(np.median(n)):,}, "
        f"90% {int(np.percentile(n, 90)):,}, max {int(n.max()):,}")
    return items


def chunks_of(it, mode, K, size):
    """windows representing one document.
    spread: the whole document if it fits in K windows (consecutive, the last one end-aligned), otherwise K evenly
    spaced windows from start to end.  first / last: the first / last K windows (section study)."""
    t = it["ids"]; n = len(t)
    if n <= size:
        return [t]
    if mode == "first":
        return [t[i * size:(i + 1) * size] for i in range(K) if i * size < n]
    if mode == "last":
        starts = sorted({max(0, n - (k + 1) * size) for k in range(K)})
        return [t[s:s + size] for s in starts]
    m = math.ceil(n / size)
    if m <= K:
        starts = [i * size for i in range(m - 1)] + [n - size]
    else:
        starts = sorted({int(round(x)) for x in np.linspace(0, n - size, K)})
    return [t[s:s + size] for s in starts]


def batch_chunks(chunk_lists, device, pad_id=0):
    flat = [c for cl in chunk_lists for c in cl]
    T = max(len(c) for c in flat)
    ids = torch.full((len(flat), T), pad_id, dtype=torch.long)
    lens = torch.zeros(len(flat), dtype=torch.long)
    for i, c in enumerate(flat):
        ids[i, :len(c)] = torch.from_numpy(c.astype(np.int64)); lens[i] = max(1, len(c))
    sl, a = [], 0
    for cl in chunk_lists:
        sl.append((a, a + len(cl))); a += len(cl)
    return ids.to(device), lens.to(device), sl


# ======================================================================================================
# 2. Model
# ======================================================================================================
class ChunkEncoder(nn.Module):
    """backbone over windows -> [learned attention pooling over tokens ; mean], on layer-normalised states (2d)"""

    def __init__(self, backbone):
        super().__init__()
        self.backbone = backbone
        self.norm = nn.LayerNorm(backbone.d)
        self.score = nn.Linear(backbone.d, 1)

    def forward(self, ids, lens):
        with nm.autocast():
            h = self.backbone(ids)
        h = self.norm(h.float())
        m = torch.arange(ids.size(1), device=ids.device)[None] < lens[:, None]
        a = torch.softmax(self.score(h).squeeze(-1).masked_fill(~m, -1e4), 1)
        att = (a[..., None] * h).sum(1)
        mean = (h * m[..., None].float()).sum(1) / lens[:, None].float()
        return torch.cat([att, mean], -1)


class LSModel(nn.Module):
    def __init__(self, backbone, n_labels, hidden=256, dropout=0.2, max_chunks=16, pos_weight=None):
        super().__init__()
        self.backbone = backbone
        self.encoder = ChunkEncoder(backbone)
        d = backbone.d
        self.proj = nn.Sequential(nn.Linear(2 * d, 2 * hidden), nn.GELU(), nn.Dropout(dropout))
        self.pos = nn.Embedding(max_chunks + 1, 2 * hidden); nn.init.zeros_(self.pos.weight)
        self.gru = nn.GRU(2 * hidden, hidden, batch_first=True, bidirectional=True)
        self.att = nn.Linear(2 * hidden, 1)
        self.head = nn.Sequential(nn.Dropout(dropout), nn.Linear(4 * hidden, 2 * hidden), nn.GELU(),
                                  nn.Dropout(dropout), nn.Linear(2 * hidden, n_labels))
        self.register_buffer("pw", pos_weight if pos_weight is not None else torch.ones(n_labels))
        self.max_chunks = max_chunks

    def doc_logits(self, cv, sl):
        out = []
        for a, b in sl:
            x = self.proj(cv[a:b])
            idx = torch.arange(b - a, device=x.device).clamp(max=self.max_chunks)
            x = x + self.pos(idx)
            h, _ = self.gru(x[None]); h = h[0]
            w = torch.softmax(self.att(h).squeeze(-1), 0)
            out.append(torch.cat([(w[:, None] * h).sum(0), h.max(0).values]))
        return self.head(torch.stack(out))

    def loss(self, logits, y, soft=None, T=2.0, alpha=0.5):
        if getattr(self, "loss_type", "bce") == "asl":       # Asymmetric Loss (Ridnik et al., ICCV 2021)
            p = torch.sigmoid(logits)
            p_neg = (p - 0.05).clamp(min=0)                     # probability shifting: ignore very easy negatives
            lp = y * (1 - p).pow(0.0) * torch.log(p.clamp(min=1e-8))
            ln = (1 - y) * p_neg.pow(4.0) * torch.log((1 - p_neg).clamp(min=1e-8))
            bce = -(lp + ln).sum(1).mean() / y.size(1) * 4.0       # per-label average (x4 keeps the scale near BCE)
        else:
            bce = F.binary_cross_entropy_with_logits(logits, y, pos_weight=self.pw)
        if soft is not None:                                   # distillation: match the teacher's softened probabilities
            kd = F.binary_cross_entropy_with_logits(logits / T, soft) * T * T
            bce = alpha * bce + (1 - alpha) * kd
        return bce


# ======================================================================================================
# 3. Metrics + thresholds
# ======================================================================================================
def f1_per_label(Y, Pb):
    tp = (Y & Pb).sum(0).astype(float); fp = (~Y & Pb).sum(0).astype(float); fn = (Y & ~Pb).sum(0).astype(float)
    p = np.divide(tp, tp + fp, out=np.zeros_like(tp), where=(tp + fp) > 0)
    r = np.divide(tp, tp + fn, out=np.zeros_like(tp), where=(tp + fn) > 0)
    f = np.divide(2 * p * r, p + r, out=np.zeros_like(tp), where=(p + r) > 0)
    return f, p, r, tp, fp, fn


def scores(Y, Pb):
    """macro-F1 over the statutes present in the reference (as the evaluator does), micro-F1, subset accuracy"""
    Y = Y.astype(bool); Pb = Pb.astype(bool)
    f, p, r, tp, fp, fn = f1_per_label(Y, Pb)
    ref = Y.sum(0) > 0
    micro_p = tp.sum() / max(1, tp.sum() + fp.sum()); micro_r = tp.sum() / max(1, tp.sum() + fn.sum())
    micro = 2 * micro_p * micro_r / max(1e-9, micro_p + micro_r)
    return dict(macro_F1=float(f[ref].mean()), macro_P=float(p[ref].mean()), macro_R=float(r[ref].mean()),
                micro_F1=float(micro), subset_acc=float((Y == Pb).all(1).mean()),
                avg_pred=float(Pb.sum(1).mean()), avg_gold=float(Y.sum(1).mean()))


def decide(P, thr, at_least_one=True):
    """probabilities (N,L) + thresholds (L,) or scalar -> boolean predictions; every document gets >= 1 statute"""
    Pb = P >= thr
    if at_least_one:
        none = ~Pb.any(1)
        if none.any():
            j = np.argmax(P[none] / np.maximum(np.broadcast_to(thr, P.shape)[none], 1e-6), 1)
            Pb[np.where(none)[0], j] = True
    return Pb


GRID = np.round(np.arange(0.02, 0.951, 0.01), 2)


def global_threshold(Y, P):
    f = [scores(Y, decide(P, t))["macro_F1"] for t in GRID]
    i = int(np.argmax(f))
    return float(GRID[i]), float(f[i])


def per_label_thresholds(Y, P, g, min_pos=5):
    """each statute with >= min_pos dev positives gets its own threshold; the others keep the global one"""
    Y = Y.astype(bool); thr = np.full(Y.shape[1], g)
    for j in range(Y.shape[1]):
        if Y[:, j].sum() >= min_pos:
            best, bt = -1.0, g
            for t in GRID:
                pb = P[:, j] >= t
                tp = (pb & Y[:, j]).sum(); fp = (pb & ~Y[:, j]).sum(); fn = (~pb & Y[:, j]).sum()
                f = 2 * tp / max(1, 2 * tp + fp + fn)
                if f > best + 1e-9: best, bt = f, t
            thr[j] = bt
    return thr


def choose_thresholds(Y, P, seed=0, log=print):
    """global vs per-statute thresholds, compared HONESTLY by 2-fold cross-fitting on dev (tune on one half,
    score on the other) -> the better strategy, refitted on the whole dev set"""
    g, fg = global_threshold(Y, P)
    rng = np.random.RandomState(seed); idx = rng.permutation(len(Y)); halves = (idx[: len(Y) // 2], idx[len(Y) // 2:])
    cv = {"global": [], "per_label": []}
    for a, b in (halves, halves[::-1]):
        ga, _ = global_threshold(Y[a], P[a])
        cv["global"].append(scores(Y[b], decide(P[b], ga))["macro_F1"])
        cv["per_label"].append(scores(Y[b], decide(P[b], per_label_thresholds(Y[a], P[a], ga)))["macro_F1"])
    cvm = {k: float(np.mean(v)) for k, v in cv.items()}
    best = max(cvm, key=cvm.get)
    thr = np.full(Y.shape[1], g) if best == "global" else per_label_thresholds(Y, P, g)
    log(f"thresholds: global {g:.2f} (dev macro-F1 {fg:.4f}) | 2-fold cross-fitted macro-F1: global {cvm['global']:.4f}, "
        f"per-statute {cvm['per_label']:.4f} -> using {best}")
    return thr, dict(global_thr=g, global_F1=fg, cv=cvm, strategy=best)


# ======================================================================================================
# 4. Training / prediction
# ======================================================================================================
@torch.no_grad()
def predict(model, items, cfg, device, enc=None):
    """-> np.float32 array (N, L) of statute probabilities"""
    model.eval(); enc = enc or model.encoder
    out = []
    bs = cfg.get("eval_docs", 8)
    for i in range(0, len(items), bs):
        cl = [chunks_of(it, cfg["mode"], cfg["K"], cfg["size"]) for it in items[i:i + bs]]
        ids, lens, sl = batch_chunks(cl, device)
        out.append(torch.sigmoid(model.doc_logits(enc(ids, lens), sl).float()).cpu().numpy())
    return np.concatenate(out).astype(np.float32)


def train(model, tr, Ytr, dv, Ydv, te, cfg, device, log=print, soft=None, on_eval=None):
    rng = random.Random(cfg["seed"])
    gpus = torch.cuda.device_count()
    enc = nn.DataParallel(model.encoder) if gpus > 1 else model.encoder
    is_bb = lambda n: n.startswith(("backbone.", "encoder.backbone."))
    named, seen = {}, set()
    for n, p in model.named_parameters():
        if p.requires_grad and id(p) not in seen:
            named[n] = p; seen.add(id(p))
    groups = [dict(params=[p for n, p in named.items() if is_bb(n)], lr=cfg["lr"]),
              dict(params=[p for n, p in named.items() if not is_bb(n)], lr=cfg["lr_head"])]
    groups = [g for g in groups if g["params"]]
    for g in groups: g["base_lr"] = g["lr"]
    params = [p for g in groups for p in g["params"]]
    opt = torch.optim.AdamW(groups, weight_decay=cfg["weight_decay"])
    scaler = torch.amp.GradScaler("cuda", enabled=torch.cuda.is_available())
    B, acc = cfg["batch_docs"], cfg["accum"]
    steps_per_epoch = math.ceil(len(tr) / (B * acc))
    total = int(steps_per_epoch * cfg["epochs"]); warm = max(1, int(cfg["warmup_frac"] * total))
    eval_every = max(1, int(steps_per_epoch * cfg.get("eval_frac", 1.0)))
    log(f"train {len(tr):,} docs | {B} docs x {acc} accum per step | {steps_per_epoch:,} steps/epoch x {cfg['epochs']} "
        f"| eval every {eval_every:,} steps | trainable {sum(p.numel() for p in params):,} | GPUs {gpus}")
    assert gpus >= 1 or cfg.get("allow_cpu"), "NO GPU in this session - set Accelerator to GPU T4 x2 (or quota is used up)"
    t0 = time.time(); budget = cfg.get("budget_hours", 7.0) * 3600
    snaps, best, best_state, step, run, nb = [], -1.0, None, 0, 0.0, 0
    K = cfg.get("snapshot_k", 2); last_eval = [-1]; hist = []; gn = [0.0, 0, 0]
    Ytr_t = torch.from_numpy(Ytr.astype(np.float32))
    if torch.cuda.is_available(): torch.cuda.reset_peak_memory_stats()

    def evaluate_now(tag):
        nonlocal best, best_state
        last_eval[0] = step
        p = predict(model, dv, cfg, device, enc)
        g, f = global_threshold(Ydv, p)
        s = scores(Ydv, decide(p, g))
        hist.append(dict(step=step, epoch=round(step / steps_per_epoch, 2), macro_F1=round(f, 4), thr=g))
        log(f"[{tag}] step {step:,} | loss {run / max(1, nb):.4f} | grad norm {gn[0] / max(1, gn[1]):.3f}, skipped {gn[2]} | "
            f"dev macro-F1 {f:.4f} @ {g:.2f} (micro {s['micro_F1']:.4f}, statutes/doc {s['avg_pred']:.2f} vs {s['avg_gold']:.2f}) | "
            f"{(time.time() - t0) / 60:.0f} min")
        if len(snaps) < K or f > min(x["f1"] for x in snaps):
            snaps.append(dict(step=step, f1=f, dev=p, test=predict(model, te, cfg, device, enc)))
            snaps.sort(key=lambda x: -x["f1"]); del snaps[K:]
        if f > best:
            best = f; best_state = {k: v.detach().to("cpu", copy=True) for k, v in model.state_dict().items()}
        if on_eval is not None:                                # interim save: a run cut off later still leaves a submission
            try:
                on_eval(np.mean([x["dev"] for x in snaps], 0), np.mean([x["test"] for x in snaps], 0), step)
            except Exception as e:
                log(f"  (interim save failed: {e})")
        model.train()

    stop = False
    for ep in range(math.ceil(cfg["epochs"])):
        model.train(); order = list(range(len(tr))); rng.shuffle(order)
        for k in range(0, len(order), B):
            idx = order[k:k + B]
            cl = [chunks_of(tr[i], cfg["mode"], cfg["K"], cfg["size"]) for i in idx]
            ids, lens, sl = batch_chunks(cl, device)
            lg = model.doc_logits(enc(ids, lens), sl)
            y = Ytr_t[idx].to(device)
            st = torch.from_numpy(soft[idx]).to(device) if soft is not None else None
            loss = model.loss(lg.float(), y, st, cfg.get("kd_T", 2.0), cfg.get("kd_alpha", 0.5)) / acc
            scaler.scale(loss).backward(); run += loss.item() * acc; nb += 1
            if (k // B + 1) % acc:
                continue
            step += 1
            f = step / warm if step <= warm else max(0.0, (total - step) / max(1, total - warm))
            for g in opt.param_groups: g["lr"] = g["base_lr"] * f
            scaler.unscale_(opt); gnorm = float(torch.nn.utils.clip_grad_norm_(params, cfg["clip"]))
            if math.isfinite(gnorm): gn[0] += gnorm; gn[1] += 1
            else: gn[2] += 1
            scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True)
            if step % eval_every == 0 or step >= total:
                evaluate_now(f"epoch {ep + 1}"); run = nb = 0; gn[:] = [0.0, 0, 0]
            if step >= total or time.time() - t0 > budget:
                stop = True
                if time.time() - t0 > budget: log("time budget reached - stopping")
                break
        if stop:
            break
    if last_eval[0] != step:
        evaluate_now("final")
    model.load_state_dict(best_state)
    peak = torch.cuda.max_memory_allocated() / 2**30 if torch.cuda.is_available() else 0.0
    dev = np.mean([x["dev"] for x in snaps], 0); test = np.mean([x["test"] for x in snaps], 0)
    log(f"averaging the best {len(snaps)} evaluations (steps {[x['step'] for x in snaps]}, dev {[round(x['f1'], 4) for x in snaps]})")
    return dict(dev=dev, test=test, best_single=best, train_minutes=round((time.time() - t0) / 60, 1),
                peak_gpu_gb=round(peak, 2), steps=step, history=hist)


# ======================================================================================================
# 5. Experiments, submission, analysis
# ======================================================================================================
PROBS = {}
_TOKS = {}


def label_space(D):
    """statutes seen in training (only these can be predicted), in a fixed order (D["LABELS"] pins it, e.g. for smoke data)"""
    if D.get("LABELS"):
        labs = list(D["LABELS"]); return labs, {l: i for i, l in enumerate(labs)}
    labs = sorted({l for d in D["train"] for l in d["labels"]}, key=lambda v: (isinstance(v, str), v))
    return labs, {l: i for i, l in enumerate(labs)}


def multi_hot(docs, L2I):
    Y = np.zeros((len(docs), len(L2I)), dtype=bool)
    for i, d in enumerate(docs):
        for l in d["labels"] or []:
            if l in L2I: Y[i, L2I[l]] = True
    return Y


def prepared(D, arch, log=print):
    key = (nm.ARCH_INFO[arch][0], len(D["train"]), len(D["dev"]), len(D["test"]))
    if key not in _TOKS:
        _TOKS[key] = {s: tokenize(D[s], arch, D["root"], workers=D.get("workers", 4), log=log) for s in ("train", "dev", "test")}
    return _TOKS[key]


def build(cfg, D, Ytr):
    bb = nm.Backbone(cfg["arch"], D["root"], bidirectional=cfg["bidirectional"], keep_layers=cfg.get("keep_layers"),
                     init=cfg.get("init", "pretrained"))
    pos, neg = Ytr.sum(0).astype(float), (len(Ytr) - Ytr.sum(0)).astype(float)
    pw = torch.tensor(np.clip(np.sqrt(neg / np.maximum(pos, 1)), 1.0, cfg.get("pos_weight_cap", 30.0)), dtype=torch.float)
    model = LSModel(bb, Ytr.shape[1], hidden=cfg["hidden"], dropout=cfg["dropout"], pos_weight=pw)
    model.loss_type = cfg.get("loss", "bce")
    desc = nm.apply_method(model, cfg)
    if model.loss_type != "bce": desc += f" | loss {model.loss_type}"
    return model, desc


def run_experiment(cfg, D, device, out_root, log=print, soft=None, keep_train=False, subsample=None):
    t_start = time.time()
    os.makedirs(os.path.join(out_root, cfg["run"]), exist_ok=True)
    LABELS, L2I = label_space(D)
    T = prepared(D, cfg["arch"], log)
    tr, dv, te = T["train"], T["dev"], T["test"]
    Ytr, Ydv = multi_hot(D["train"], L2I), multi_hot(D["dev"], L2I)
    if subsample:
        sel = sorted(random.Random(0).sample(range(len(tr)), min(subsample, len(tr))))
        tr = [tr[i] for i in sel]; Ytr = Ytr[sel]; soft = soft[sel] if soft is not None else None
    torch.manual_seed(cfg["seed"]); random.seed(cfg["seed"]); np.random.seed(cfg["seed"])
    model, desc = build(cfg, D, Ytr)
    model.to(device)
    total = sum(p.numel() for p in model.parameters()); trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    log(f"\n=== {cfg['run']} | {cfg['arch']} | {desc} | windows: {cfg['mode']} K={cfg['K']} x {cfg['size']} tokens | "
        f"{len(LABELS)} statutes | params {total:,}, trainable {trainable:,} ({trainable / total:.2%})" + (" | KD" if soft is not None else ""))
    run_dir = os.path.join(out_root, cfg["run"])

    def on_eval(p_dev, p_test, step):
        thr_i, ti = choose_thresholds(Ydv, p_dev, log=lambda *a: None)
        f_i = scores(Ydv, decide(p_dev, thr_i))["macro_F1"]
        torch.save({cfg["run"]: dict(dev=p_dev.astype(np.float16), test=p_test.astype(np.float16), arch=cfg["arch"],
                                      method=cfg["method"], labels=LABELS, thr=thr_i, dev_F1=f_i, params=total,
                                      dev_inference_sec=0.0, interim_step=step)},
                   os.path.join(out_root, f"p5_probs_{cfg['run']}_interim.pt"))
        write_submission(p_test, thr_i, D, LABELS, os.path.join(run_dir, "interim"), cfg["run"] + "_interim",
                         log=lambda m: log("  interim save: " + m + f" | dev macro-F1 {f_i:.4f} ({ti['strategy']} thresholds)"))

    res = train(model, tr, Ytr, dv, Ydv, te, cfg, device, log, soft, on_eval=on_eval)
    t0 = time.time(); _ = predict(model, dv, cfg, device); infer = time.time() - t0
    thr, tinfo = choose_thresholds(Ydv, res["dev"], log=log)
    sc = scores(Ydv, decide(res["dev"], thr))
    log(f"DEV: macro-F1 {sc['macro_F1']:.4f} | macro P {sc['macro_P']:.4f} R {sc['macro_R']:.4f} | micro-F1 {sc['micro_F1']:.4f} | "
        f"subset acc {sc['subset_acc']:.4f} | statutes/doc {sc['avg_pred']:.2f} (gold {sc['avg_gold']:.2f})")
    PROBS[cfg["run"]] = dict(dev=res["dev"].astype(np.float16), test=res["test"].astype(np.float16), arch=cfg["arch"],
                             method=cfg["method"], labels=LABELS)
    if keep_train:
        log("predicting the training documents (teacher targets)...")
        dp = nn.DataParallel(model.encoder) if torch.cuda.device_count() > 1 else None
        PROBS[cfg["run"]]["train"] = predict(model, T["train"], cfg, device, dp).astype(np.float16)
    zp = write_submission(res["test"], thr, D, LABELS, os.path.join(out_root, cfg["run"]), cfg["run"], log)
    rec = dict(run=cfg["run"], arch=cfg["arch"], method=desc, cfg=cfg, params_total=total, params_trainable=trainable,
               dev=sc, dev_F1=sc["macro_F1"], thresholds=tinfo, dev_inference_sec=round(infer, 1), kd=soft is not None,
               zip=os.path.basename(zp), minutes_total=round((time.time() - t_start) / 60, 1), train_docs=len(tr),
               **{k: v for k, v in res.items() if k not in ("dev", "test")})
    PROBS[cfg["run"]]["thr"] = thr
    json.dump(rec, open(os.path.join(out_root, cfg["run"], "record.json"), "w"), indent=1, default=str)
    del model; torch.cuda.empty_cache()
    return rec


def write_submission(P_test, thr, D, LABELS, out_dir, name, log=print):
    """[{"id": <id as in test.json>, "pred": [unique valid statute ids]}, ...] -> prediction.json -> zip"""
    Pb = decide(np.asarray(P_test, dtype=np.float32), thr)
    assert Pb.shape[0] == len(D["test"])
    sub = [dict(id=d["id"], pred=sorted({LABELS[j] for j in np.where(row)[0]}, key=lambda v: (isinstance(v, str), v)))
           for d, row in zip(D["test"], Pb)]
    valid = set(LABELS)
    assert len({json.dumps(s["id"]) for s in sub}) == len(sub) and all(set(s["pred"]) <= valid and s["pred"] for s in sub)
    os.makedirs(out_dir, exist_ok=True)
    pj = os.path.join(out_dir, "prediction.json"); json.dump(sub, open(pj, "w"), ensure_ascii=False)
    zp = os.path.join(out_dir, f"submission_{name}.zip")
    with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
        z.write(pj, arcname="prediction.json")
    log(f"{name}: {len(sub):,} test documents | {np.mean([len(s['pred']) for s in sub]):.2f} statutes/doc -> {zp}")
    return zp


def save_probs(path, records):
    out = {r["run"]: dict(PROBS[r["run"]], dev_F1=r["dev_F1"], params=r["params_total"],
                          dev_inference_sec=r["dev_inference_sec"]) for r in records if r["run"] in PROBS}
    torch.save(out, path)


def load_probs(pattern="/kaggle/input/**/p5_probs_*.pt", log=print):
    import glob
    out = {}
    for f in sorted(glob.glob(pattern, recursive=True)):
        for n, v in torch.load(f, map_location="cpu", weights_only=False).items():
            out.setdefault(n, v)
        log(f"loaded {f}")
    return out


def analysis(D, P_dev, thr, LABELS, log=print, top=12):
    """rare vs frequent statutes, most missed / over-predicted statutes, co-occurring pairs, cardinality"""
    L2I = {l: i for i, l in enumerate(LABELS)}
    Ytr, Ydv = multi_hot(D["train"], L2I), multi_hot(D["dev"], L2I)
    Pb = decide(P_dev, thr)
    f, p, r, tp, fp, fn = f1_per_label(Ydv, Pb)
    freq = Ytr.sum(0); ref = Ydv.sum(0) > 0
    qs = np.percentile(freq[ref], [0, 33, 66, 100])
    for name, lo, hi in (("rare", qs[0], qs[1]), ("medium", qs[1], qs[2]), ("frequent", qs[2], qs[3] + 1)):
        m = ref & (freq >= lo) & (freq < hi) if name != "frequent" else ref & (freq >= lo)
        if m.any():
            log(f"{name:<9} statutes ({int(m.sum())}, {int(lo)}-{int(hi)} training docs): dev macro-F1 {f[m].mean():.4f} | P {p[m].mean():.3f} R {r[m].mean():.3f}")
    order = np.argsort(-(fn))
    log("most missed statutes (id: missed / dev positives, F1): " + ", ".join(
        f"{LABELS[j]}: {int(fn[j])}/{int(Ydv[:, j].sum())} F1 {f[j]:.2f}" for j in order[:top] if Ydv[:, j].sum()))
    order = np.argsort(-(fp))
    log("most over-predicted statutes (id: false positives, F1): " + ", ".join(
        f"{LABELS[j]}: {int(fp[j])} F1 {f[j]:.2f}" for j in order[:top]))
    co = Counter()
    for row in Ytr:
        js = np.where(row)[0]
        for a in range(len(js)):
            for b in range(a + 1, len(js)): co[(js[a], js[b])] += 1
    pairs = co.most_common(top)
    log("most frequent co-occurring statute pairs in training (pair: count | dev: both gold -> both predicted %): " + ", ".join(
        f"{LABELS[a]}+{LABELS[b]}: {c} | {int(((Ydv[:, a] & Ydv[:, b]) & (Pb[:, a] & Pb[:, b])).sum())}/{int((Ydv[:, a] & Ydv[:, b]).sum())}"
        for (a, b), c in pairs))
    k_gold = Ydv.sum(1)
    for k in (1, 2, 3):
        m = (k_gold == k) if k < 3 else (k_gold >= 3)
        if m.any():
            s = scores(Ydv[m], Pb[m])
            log(f"documents with {k if k < 3 else '3+'} gold statutes ({int(m.sum())}): micro-F1 {s['micro_F1']:.4f}, subset acc {s['subset_acc']:.4f}, predicted/doc {s['avg_pred']:.2f}")


def bagged_thresholds(Y, P, g, n_bags=15, seed=0):
    """per-statute thresholds averaged over many random halves of the data (lower variance than one fit)"""
    rng = np.random.RandomState(seed); acc = np.zeros(Y.shape[1])
    for _ in range(n_bags):
        h = rng.choice(len(Y), len(Y) // 2, replace=False)
        acc += per_label_thresholds(Y[h], P[h], g)
    return acc / n_bags


def choose_thresholds3(Y, P, seed=0, log=print):
    """global vs per-statute vs bagged per-statute thresholds, compared by 2-fold cross-fitting on dev"""
    g, fg = global_threshold(Y, P)
    rng = np.random.RandomState(seed); idx = rng.permutation(len(Y)); halves = (idx[: len(Y) // 2], idx[len(Y) // 2:])
    cv = {"global": [], "per_label": [], "bagged": []}
    for a, b in (halves, halves[::-1]):
        ga, _ = global_threshold(Y[a], P[a])
        cv["global"].append(scores(Y[b], decide(P[b], ga))["macro_F1"])
        cv["per_label"].append(scores(Y[b], decide(P[b], per_label_thresholds(Y[a], P[a], ga)))["macro_F1"])
        cv["bagged"].append(scores(Y[b], decide(P[b], bagged_thresholds(Y[a], P[a], ga, seed=seed)))["macro_F1"])
    cvm = {k: float(np.mean(v)) for k, v in cv.items()}
    best = max(cvm, key=cvm.get)
    thr = {"global": lambda: np.full(Y.shape[1], g), "per_label": lambda: per_label_thresholds(Y, P, g),
           "bagged": lambda: bagged_thresholds(Y, P, g, seed=seed)}[best]()
    log(f"thresholds (cross-fitted dev macro-F1): global {cvm['global']:.4f} | per-statute {cvm['per_label']:.4f} | "
        f"bagged per-statute {cvm['bagged']:.4f} -> using {best}")
    return thr, dict(global_thr=g, global_F1=fg, cv=cvm, strategy=best)

In [ ]:
import importlib
sys.path.insert(0, os.path.abspath("p5src"))
import ner_data as nd, ner_model as nm, ls
for _m in (nd, nm, ls): importlib.reload(_m)
assert getattr(ls, "VERSION", "") == "p5-v3", "OLD CODE - import the Phase-5 notebook file"
print("Phase-5 code imported | code version:", ls.VERSION)

In [ ]:
# 4 · Load Phase_5_LSI: train / dev / test, with checks and statistics (incl. a dev/test overlap check)
from collections import Counter
from huggingface_hub import HfApi, hf_hub_download
REPO = "Exploration-Lab/CS-781-Capstone"
files = [f for f in HfApi(token=HF_TOKEN).list_repo_files(REPO, repo_type="dataset") if f.startswith("Phase_5")]
print("\n".join(files))
get = lambda n: hf_hub_download(REPO, next(f for f in files if f.endswith("/" + n)), repo_type="dataset", token=HF_TOKEN)
D = dict(root=ROOT, workers=min(4, os.cpu_count() or 1), train=[])
SPLITS = globals().get("SPLITS", ("train", "dev", "test"))
for split in SPLITS:
    D[split] = ls.parse(ls.read_json_any(get(f"{split}.json")))
    w = np.array([len(d["text"].split()) for d in D[split]])
    k = [len(d["labels"]) for d in D[split] if d["labels"]]
    print(f"{split:<5}: {len(D[split]):6,} documents | words median {int(np.median(w)):,}, 90% {int(np.percentile(w, 90)):,}, "
          f"max {int(w.max()):,} | statutes/doc {np.mean(k) if k else 0:.2f} | labelled docs {len(k):,}")
for split in [s for s in ("train", "dev") if s in SPLITS]:
    assert all(d["labels"] for d in D[split]), f"{split}: every document needs >= 1 statute"
assert len({json.dumps(d["id"]) for d in D["test"]}) == len(D["test"]), "duplicate test ids"
LABELS, L2I = ls.label_space(D) if D["train"] else (None, None)
D["LABELS"] = LABELS                       # fixed statute order for every run (and the smoke data)
if D["train"]:
    fr = Counter(l for d in D["train"] for l in d["labels"])
    v = np.array(sorted(fr.values()))
    dev_l = {l for d in D["dev"] for l in d["labels"]}
    print(f"statutes: {len(LABELS)} in train (frequency min {v.min()}, median {int(np.median(v))}, max {v.max()}) | {len(dev_l)} in dev, "
          f"{len(dev_l - set(LABELS))} of them never in train | label type {type(LABELS[0]).__name__} | examples {LABELS[:8]}")
pre = lambda d: d["text"][:1500]
tr_p = {pre(d) for d in D["train"]}; dv_p = {pre(d) for d in D["dev"]}
dv_ids = {json.dumps(d["id"]) for d in D["dev"]}
print("OVERLAP CHECK | dev texts in train:", sum(pre(d) in tr_p for d in D["dev"]),
      "| test texts in train:", sum(pre(d) in tr_p for d in D["test"]),
      "| test texts in dev:", sum(pre(d) in dv_p for d in D["test"]),
      "| test ids in dev:", sum(json.dumps(d["id"]) in dv_ids for d in D["test"]))
print("test documents that carry labels in test.json:", sum(1 for d in D["test"] if d["labels"]), "(never used)")
ex = (D["train"] or D["dev"])[0]
print(f"\nexample {ex['id']} statutes {ex['labels']}\n  START: {ex['text'][:400]}\n  END  : ...{ex['text'][-300:]}")
rnd = random.Random(0)
D_SMOKE = dict(D, train=rnd.sample(D["train"], min(24, len(D["train"]))) if D["train"] else [],
               dev=rnd.sample(D["dev"], 16), test=D["test"][:16])

In [ ]:
# 6 · Final ensemble over everything available (finished runs AND interim epoch files) -> top-2 zips
import itertools
M = ls.load_probs()
Ydv = ls.multi_hot(D["dev"], L2I)
names = [n for n in sorted(M) if "student" not in n and M[n]["arch"] in ("arch4_mla", "arch3_prefixlm")
         and list(M[n]["labels"]) == list(LABELS)]
for n in names:
    print(f"{n:<40} {M[n]['arch']:<16} dev macro-F1 {M[n]['dev_F1']:.4f}" + (f"  (interim, step {M[n]['interim_step']})" if "interim_step" in M[n] else ""))
short = lambda n: n.split("_")[1]
avg = lambda c, k: np.mean([M[n][k].astype(np.float32) for n in c], 0)
quick = []
for k in range(1, len(names) + 1):
    for c in itertools.combinations(names, k):
        quick.append((ls.global_threshold(Ydv, avg(c, "dev"))[1], c))
quick.sort(key=lambda x: -x[0])
print("\ntop combinations (global threshold):", [("+".join(short(n) for n in c), round(f, 4)) for f, c in quick[:8]])
R = []
for f, c in quick[:6]:
    print("\n" + "+".join(short(n) for n in c))
    thr, ti = ls.choose_thresholds3(Ydv, avg(c, "dev"))
    R.append(dict(c=c, thr=thr, cv=ti["cv"][ti["strategy"]], strategy=ti["strategy"]))
R.sort(key=lambda r: -r["cv"])
print(f"\n{'members':<40}{'cross-fitted dev F1':>21}{'thresholds':>12}")
for r in R: print(f"{'+'.join(short(n) for n in r['c']):<40}{r['cv']:>21.4f}{r['strategy']:>12}")
for i, r in enumerate(R[:2]):
    tag = "_".join(short(n) for n in r["c"])
    ls.write_submission(avg(r["c"], "test"), r["thr"], D, LABELS, os.path.join(OUT, f"final{i + 1}_{tag}"), f"p5_final{i + 1}_{tag}")
print("\nSUBMIT final1 first (best honest dev score); final2 is the runner-up.")

In [ ]:
# 7 · (nothing to train)

In [ ]:
# 8 · (see cell 6)